<a href="https://colab.research.google.com/github/Koji-Accounting-Tech/D365-Finance-Data-Validator/blob/main/D365_Data_Processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# 1. Excelファイルの読み込み
df = pd.read_excel('test_data.xlsx')

# 2. データの先頭5行を表示して中身を確認
print("--- データの先頭5行 ---")
display(df.head())

# 3. 基本的な統計量（件数、合計、平均など）を表示
print("\n--- データの統計情報 ---")
display(df.describe())

--- データの先頭5行 ---


,伝票番号,日付,勘定科目コード,借方金額,貸方金額,摘要
0,1,2026-04-01,100501,100000,100000,備品
1,2,2026-04-01,100601,200000,200000,買掛金支払
2,3,2026-04-01,100701,300000,300000,売掛金入金
3,4,2026-04-02,100802,400000,400000,減価償却・設備
4,5,2026-04-02,200101,500000,500000,売上



--- データの統計情報 ---


,伝票番号,日付,勘定科目コード,借方金額,貸方金額
count,5.000000,5,5.000000,5.000000,5.000000
mean,3.000000,2026-04-01 09:36:00,120541.200000,300000.000000,300000.000000
min,1.000000,2026-04-01 00:00:00,100501.000000,100000.000000,100000.000000
25%,2.000000,2026-04-01 00:00:00,100601.000000,200000.000000,200000.000000
50%,3.000000,2026-04-01 00:00:00,100701.000000,300000.000000,300000.000000
75%,4.000000,2026-04-02 00:00:00,100802.000000,400000.000000,400000.000000
max,5.000000,2026-04-02 00:00:00,200101.000000,500000.000000,500000.000000
std,1.581139,NaN,44475.421642,158113.883008,158113.883008


In [4]:
import pandas as pd

def calculate_straight_line_depreciation(acquisition_cost, salvage_value, useful_life_years):
    """
    定額法の減価償却スケジュールを計算する関数
    """
    # 年間の償却額 = (取得価額 - 残存価額) / 耐用年数
    annual_depreciation = (acquisition_cost - salvage_value) / useful_life_years

    schedule = []
    current_book_value = acquisition_cost

    for year in range(1, useful_life_years + 1):
        # 最終年の調整（残存価額を下回らないようにする）
        if year == useful_life_years:
            depreciation_amount = current_book_value - salvage_value
        else:
            depreciation_amount = annual_depreciation

        current_book_value -= depreciation_amount

        schedule.append({
            "年": year,
            "期首帳簿価額": round(current_book_value + depreciation_amount),
            "償却額": round(depreciation_amount),
            "期末帳簿価額": round(current_book_value)
        })

    return pd.DataFrame(schedule)

# --- テスト用の設定値 ---
cost = 1200000    # 取得価額（120万円）
salvage = 10000   # 残存価額（1万円）
life = 5          # 耐用年数（5年）

# 実行
depreciation_df = calculate_straight_line_depreciation(cost, salvage, life)

print(f"取得価額: {cost:,}円 / 耐用年数: {life}年 の減価償却スケジュール")
display(depreciation_df)

# Excelとして保存したい場合（任意）
# depreciation_df.to_excel("depreciation_schedule.xlsx", index=False)

取得価額: 1,200,000円 / 耐用年数: 5年 の減価償却スケジュール


,年,期首帳簿価額,償却額,期末帳簿価額
0,1,1200000,238000,962000
1,2,962000,238000,724000
2,3,724000,238000,486000
3,4,486000,238000,248000
4,5,248000,238000,10000


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

print("Current working directory files:")
for dirname, _, filenames in os.walk('.'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

Current working directory files:
./journal_test.xlsx
./test_data_2.xlsx
./test_data.xlsx
./.config/active_config
./.config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
./.config/gce
./.config/.last_opt_in_prompt.yaml
./.config/.last_survey_prompt.yaml
./.config/config_sentinel
./.config/default_configs.db
./.config/.last_update_check.json
./.config/logs/2026.03.30/13.29.11.295522.log
./.config/logs/2026.03.30/13.28.36.228080.log
./.config/logs/2026.03.30/13.29.00.305433.log
./.config/logs/2026.03.30/13.29.12.938774.log
./.config/logs/2026.03.30/13.29.24.176304.log
./.config/logs/2026.03.30/13.29.24.988708.log
./.config/configurations/config_default
./sample_data/README.md
./sample_data/anscombe.json
./sample_data/mnist_train_small.csv
./sample_data/mnist_test.csv
./sample_data/california_housing_test.csv
./sample_data/california_housing_train.csv


In [ ]:
import pandas as pd

# Excelファイルを読み込む
df = pd.read_excel('test_data_2.xlsx')

# 現在読み込んでいるデータの列名をすべて表示する
print("--- 現在のExcelの列名一覧 ---")
print(df.columns.tolist())

# 列のデータ型や欠損値もまとめて確認する
print("\n--- データの詳細情報 ---")

# 1. 貸借一致チェック（会計コンサルの基本！）
debit_total = df['借方金額'].sum()
credit_total = df['貸方金額'].sum()

print(f"--- 貸借チェック結果 ---")
if debit_total == credit_total:
    print(f"✅ OK: 貸借が一致しています（合計: {debit_total}）")
else:
    print(f"❌ NG: 貸借が不一致です（借方: {debit_total}, 貸方: {credit_total}）")

# 2. 必須項目の空欄（欠損値）チェック
print("\n--- 必須項目チェック ---")
missing_accounts = df['勘定科目コード'].isnull().sum()
if missing_accounts > 0:
    print(f"⚠️ 警告: 勘定科目コードが入力されていない行が {missing_accounts} 件あります。")
else:
    print("✅ OK: すべての行に勘定科目コードが入っています。")

--- 現在のExcelの列名一覧 ---
['伝票番号', '日付', '勘定科目コード', '借方金額', '貸方金額', '摘要']

--- データの詳細情報 ---
--- 貸借チェック結果 ---
✅ OK: 貸借が一致しています（合計: 1500000）

--- 必須項目チェック ---
✅ OK: すべての行に勘定科目コードが入っています。
